# PPMS Measurements

This notebook was created to control our electronics during measurements using the MRSEC PPMS.
Because it's a shared cryostat, we have to bring a computer in to the room for control, so try to keep the code here lightweight and self-contained.

Currently, this notebook is configured for our measurements using a single lockin amplifier (MFLI, Zurich Instruments) and two Keithley 2450 SourceMeters.

## Device Info

**MFLI info**

| Device | Serial | Options | IP Address |
| :--- | :--- | :--- | :--- |
| **MFLI 1** | DEV32203 | MD | 192.168.1.20 |
| **MFLI 2** | DEV32269 | *None* | 192.168.1.21 |

**Keithley info**

| Device | Serial | Options | IP Address |
| :--- | :--- | :--- | :--- |
| **Keithley 1** | MYFP001474 | *None* | 192.168.1.30 |
| **Keithley 2** | 04670993 | *None* | 192.168.1.31 |

---
# Setup
---

#### Helper Functions and Dependencies

In [ ]:
import qcodes as qc
import numpy as np
import os
from datetime import datetime
from zhinst.qcodes import ZISession, MFLI
from qcodes.instrument_drivers.Keithley import Keithley2450
import time
from tqdm import tqdm
from qcodes.dataset import (
    Measurement,
    initialise_or_create_database_at,
    load_by_guid,
    load_by_run_spec,
    load_or_create_experiment,
    plot_dataset,
)

from kmeasure import ramp
from kmeasure.tools import MFLIDemodPoller, PPMSClient, PPMSInstrument

### QCoDeS Station

In [ ]:
# initialize qcodes station
qc.Instrument.close_all()
station = qc.Station()
station.close_all_registered_instruments()

### Connect Lockin

Instantiate a zhinst.toolkit session (using QCoDeS wrapper). Edit `mfli_serial` if you want to change which MFLI you use.  

In [ ]:
# MFLI info here
mfli_serial = 'dev32203'
interface = '1GbE'
server_host = 'localhost'

try:
    # instantiate zhinst.toolkit session using the qcodes zhinst.toolkit wrapper (zhinst.qcodes), 
    # create mfli instrument object, and add this to the qcodes station
    session = ZISession(server_host, allow_version_mismatch=True)
    mfli = session.connect_device(serial=mfli_serial, interface=interface)
    station.add_component(mfli)
    print('Connected to MFLI')
    
except Exception as e:
    print(e)
    print('Error connecting to MFLI. If you opened LabOne and interfaced with this MFLI before: (1) Close LabOne. (2) Power cycle the MFLI (3) Wait about 20s after the light turns blue. (4) Rerun this cell')

Connected to MFLI


To close the mfli run `mfli.close`.

In [ ]:
# mfli.close()

Once you've connected to the MFLI, you can reopen LabOne via the IP address (printed on MFLI) or any other way :). One time we had a problem with using both the API and the software at the same time. This was fixed by updating firmware. 

### Connect Keithleys
Running this cell will take a second. Due to some QCoDeS issue we get a lot of errors like:
`Could not update parameter: user_delay` and `Could not update parameter: sweep_axis`. 
Just don't worry about these. They don't matter.
Running this cell should take ~1 minute or less. If it hangs longer than that, you may have an issue. 

In [ ]:
# Keithley sourcemeters

keithley_1_ip_addr = '192.168.1.30'
keithley_2_ip_addr = '192.168.1.31'

keithley_1_visa_addr = f'TCPIP::{keithley_1_ip_addr}::5025::SOCKET'
keithley_2_visa_addr = f'TCPIP::{keithley_2_ip_addr}::5025::SOCKET'

# instantiate and add Keithleys to station 

keithley_1 = Keithley2450("keithley_1", keithley_1_visa_addr)
keithley_2 = Keithley2450("keithley_2", keithley_2_visa_addr)

station.add_component(keithley_1)
station.add_component(keithley_2)
print('Connected to Keithleys.')

###
# set up and alias keithleys

keithley_1.terminals('front')
keithley_2.terminals('front')
keithley_1.sense.four_wire_measurement(False)
keithley_2.sense.four_wire_measurement(False)

keithley_1.sense.function('current')
keithley_2.sense.function('current')

# set sense range to 100 mA 
# this sets the internal settling time of the Keithley to something acceptable for remote control
keithley_1.sense.range(1e-1)
keithley_2.sense.range(1e-1)

keithley_1.source.function('voltage')
keithley_2.source.function('voltage')

# alias sweep params
V_g = keithley_1.source.voltage
V_b = keithley_2.source.voltage

back_gate = keithley_1
bias = keithley_2

[keithley_1_source(Keithley2450Source)] Snapshot: Could not update parameter: sweep_axis


Connected to: KEITHLEY INSTRUMENTS 2450 (serial:MYFP001474, firmware:1.7.16a) in 0.14s
Connected to: KEITHLEY INSTRUMENTS 2450 (serial:04670993, firmware:1.7.16a) in 0.12s


[keithley_1_source(Keithley2450Source)] Snapshot: Could not update parameter: user_delay
[keithley_1_source(Keithley2450Source)] Snapshot: Could not update parameter: sweep_axis
[keithley_1_source(Keithley2450Source)] Snapshot: Could not update parameter: user_delay
[keithley_1_sense(Keithley2450Sense)] Snapshot: Could not update parameter: user_delay
[keithley_1_sense(Keithley2450Sense)] Snapshot: Could not update parameter: user_delay
[keithley_1_sense(Keithley2450Sense)] Snapshot: Could not update parameter: user_delay
[keithley_2_source(Keithley2450Source)] Snapshot: Could not update parameter: sweep_axis
[keithley_2_source(Keithley2450Source)] Snapshot: Could not update parameter: user_delay
[keithley_2_source(Keithley2450Source)] Snapshot: Could not update parameter: sweep_axis
[keithley_2_source(Keithley2450Source)] Snapshot: Could not update parameter: user_delay
[keithley_2_sense(Keithley2450Sense)] Snapshot: Could not update parameter: user_delay
[keithley_2_sense(Keithley245

Connected to Keithleys.


To close the mfli run `mfli.close`.

In [8]:
# keithley_1.close()

### Connect PPMS Client

Don't really want to use this. Need a find a different way to connect to PPMS. 

In [ ]:
ppms = PPMSInstrument('ppms')
ppms.client.manager.conn.send('Hello PPMS'.encode('utf-8'))
ppms.client.manager.conn.recv(4096)

In [ ]:
ppms.client.manager.conn.recv(4096)

10

In [1]:
# ppms.temperature.get()
# ppms.field.get()
# ppms.client.set_temp(1.8,4,0)
# ppms.temperature_status.get()

# field_setpoint = 90000
# rate = 30
# approach_mode = 0 # 0 is linear
# end_mode = 0 # persistent

# B = ppms.field

# ppms.client.set_field(field_setpoint, rate, approach_mode, end_mode)

# setpoints = [900, 0]

# experiment_name = 'test'
# sample_name = 'testing_B_sweep'

# exp = load_or_create_experiment(
#     experiment_name = experiment_name,
#     sample_name = sample_name
# )

# meas = Measurement(exp=exp, name="B_sweep")

# meas.register_parameter(B)

# meas.write_period = 0.1 # read from cache every X seconds

# total_time_allowed = 300
# wait_time = 30
# N_tries_for_field = int(total_time_allowed/wait_time)

# with meas.run() as datasaver:
#     # ↓ main scan loop ↓
#     for itr, setpoint in enumerate(tqdm(setpoints)):
#         ppms.client.set_field(setpoint, rate, approach_mode, end_mode)
#         for i in range(N_tries_for_field):
#             if i == N_tries_for_field - 1:
#                 raise Exception(f'Did not reach field setpoint in {total_time_allowed/60} min. Raising exception and stopping scan.')
#             time.sleep(wait_time)
#             if ppms.field_status.get() == "Persistent":
#                 print(f'Persistent at {setpoint} Oe in {i+1} {wait_time}s intervals.')
#                 break
#             else:
#                 print(f'[Status] {ppms.field_status.get()}')
#                 continue
            
#         datasaver.add_result(
#             (B, B.get()), 
#         )

### Other Station Components

Static, manually adjusted (e.g. Femto preamp), station components here.

In [2]:
v_div_AC_side_resistance = 40e3
v_div_DC_side_resistance = 10e3

preamp_gain = 1e3

## StudentBot 🤖
I am a robot.

In [25]:
# 260728 need to restart kernel to run this
from PPMS_studentbot.app import StudentBot

# @StudentBot status_keyword in a Slack channel will return status
status_keyword = 'PPMS'
studentbot = StudentBot(status_keyword)
# use set_status method to update the status 
studentbot.set_status('StudentBot status not set yet. Use set_status() method.')
studentbot.start()

# Use .stop() method to stop. 

# StudentBot runs on a separate daemonic thread. 
# That means that you can run StudentBot in the background without
# interrupting any of the code in this notebook, but killing
# this kernel will still kill the StudentBot thread.

Bot initialized as User ID: U0BKYCYCJSF
StudentBot monitoring 'PPMS' on a background thread.


INFO:slack_bolt.App:A new session has been established (session id: fc162cd2-2ac0-4e61-a4a5-021808cc3436)
INFO:slack_bolt.App:Starting to receive messages from a new connection (session id: fc162cd2-2ac0-4e61-a4a5-021808cc3436)


DEBUG Event Received: {'type': 'message', 'user': 'U0BHW7P6D7C', 'ts': '1785277718.714169', 'client_msg_id': 'A00205A8-B48F-4225-B76D-2C1A47F9ED4C', 'text': '<@U0BKYCYCJSF> PPMS', 'team': 'T0916MXAW2F', 'blocks': [{'type': 'rich_text', 'block_id': 'DEfZ6', 'elements': [{'type': 'rich_text_section', 'elements': [{'type': 'user', 'user_id': 'U0BKYCYCJSF'}, {'type': 'text', 'text': ' PPMS'}]}]}], 'channel': 'D0BL2LNFFJA', 'event_ts': '1785277718.714169', 'channel_type': 'im'}
DEBUG Event Received: {'type': 'message', 'user': 'U0BHW7P6D7C', 'ts': '1785280845.946009', 'client_msg_id': '4a5f3c5a-df03-48be-a8e3-620424ab737b', 'text': '<@U0BKYCYCJSF> PPMS', 'team': 'T0916MXAW2F', 'blocks': [{'type': 'rich_text', 'block_id': '58Mfi', 'elements': [{'type': 'rich_text_section', 'elements': [{'type': 'user', 'user_id': 'U0BKYCYCJSF'}, {'type': 'text', 'text': ' PPMS'}]}]}], 'channel': 'C0BHF6H84G7', 'event_ts': '1785280845.946009', 'channel_type': 'group'}
DEBUG Event Received: {'type': 'message',

INFO:slack_bolt.App:Connecting to a new endpoint...
INFO:slack_bolt.App:The connection has been closed (session id: fc162cd2-2ac0-4e61-a4a5-021808cc3436)
INFO:slack_bolt.App:A new session has been established (session id: 17f33527-50c6-4947-915f-df9127d5c968)
INFO:slack_bolt.App:Connected to a new endpoint...
INFO:slack_bolt.App:Stopped receiving messages from a connection (session id: fc162cd2-2ac0-4e61-a4a5-021808cc3436)
INFO:slack_bolt.App:Starting to receive messages from a new connection (session id: 17f33527-50c6-4947-915f-df9127d5c968)


DEBUG Event Received: {'type': 'message', 'user': 'U0BJQH6GF88', 'ts': '1785296609.816919', 'client_msg_id': '61D00503-3529-4BBA-B8C5-4178A4E4EA20', 'text': 'Haha ', 'team': 'T0916MXAW2F', 'blocks': [{'type': 'rich_text', 'block_id': 'VApYZ', 'elements': [{'type': 'rich_text_section', 'elements': [{'type': 'text', 'text': 'Haha '}]}]}], 'channel': 'C0BHF6H84G7', 'event_ts': '1785296609.816919', 'channel_type': 'group'}
DEBUG Event Received: {'type': 'message', 'user': 'U0BJQH6GF88', 'ts': '1785296612.269419', 'client_msg_id': '8BD7E54F-EB00-4DE9-AA55-A28C0D2B120D', 'text': 'Welp ', 'team': 'T0916MXAW2F', 'blocks': [{'type': 'rich_text', 'block_id': 'GywqZ', 'elements': [{'type': 'rich_text_section', 'elements': [{'type': 'text', 'text': 'Welp '}]}]}], 'channel': 'C0BHF6H84G7', 'event_ts': '1785296612.269419', 'channel_type': 'group'}


INFO:slack_bolt.App:Connecting to a new endpoint...
INFO:slack_bolt.App:The connection has been closed (session id: 17f33527-50c6-4947-915f-df9127d5c968)
INFO:slack_bolt.App:A new session has been established (session id: 9770df49-2dff-4fe5-8026-0a2dd11b7a13)
INFO:slack_bolt.App:Connected to a new endpoint...
INFO:slack_bolt.App:Stopped receiving messages from a connection (session id: 17f33527-50c6-4947-915f-df9127d5c968)
INFO:slack_bolt.App:Starting to receive messages from a new connection (session id: 9770df49-2dff-4fe5-8026-0a2dd11b7a13)


DEBUG Event Received: {'type': 'message', 'user': 'U0BHW7P6D7C', 'ts': '1785330209.634449', 'client_msg_id': '06FEDAAB-7869-4244-A5AC-C2461C3B03DD', 'text': '<@U0BKYCYCJSF> PPMS', 'team': 'T0916MXAW2F', 'blocks': [{'type': 'rich_text', 'block_id': 'DEfZ6', 'elements': [{'type': 'rich_text_section', 'elements': [{'type': 'user', 'user_id': 'U0BKYCYCJSF'}, {'type': 'text', 'text': ' PPMS'}]}]}], 'channel': 'C0BHF6H84G7', 'event_ts': '1785330209.634449', 'channel_type': 'group'}
DEBUG Event Received: {'type': 'message', 'user': 'U0BJQH6GF88', 'ts': '1785330691.835409', 'client_msg_id': 'CD1BFB5A-0E2D-47FD-85ED-320AB8C3D31D', 'text': 'Omg what it talks to me ', 'team': 'T0916MXAW2F', 'blocks': [{'type': 'rich_text', 'block_id': 'Ggvsu', 'elements': [{'type': 'rich_text_section', 'elements': [{'type': 'text', 'text': 'Omg what it talks to me '}]}]}], 'channel': 'C0BHF6H84G7', 'event_ts': '1785330691.835409', 'channel_type': 'group'}
DEBUG Event Received: {'type': 'message', 'subtype': 'thre

INFO:slack_bolt.App:Connecting to a new endpoint...
INFO:slack_bolt.App:The connection has been closed (session id: 9770df49-2dff-4fe5-8026-0a2dd11b7a13)
INFO:slack_bolt.App:A new session has been established (session id: 78b0a335-e654-4194-b336-66b6d8b40b98)
INFO:slack_bolt.App:Connected to a new endpoint...
INFO:slack_bolt.App:Stopped receiving messages from a connection (session id: 9770df49-2dff-4fe5-8026-0a2dd11b7a13)
INFO:slack_bolt.App:Starting to receive messages from a new connection (session id: 78b0a335-e654-4194-b336-66b6d8b40b98)


DEBUG Event Received: {'type': 'message', 'subtype': 'thread_broadcast', 'user': 'U0BHW7P6D7C', 'thread_ts': '1785331103.419059', 'root': {'user': 'U0BJQH6GF88', 'type': 'message', 'ts': '1785331103.419059', 'client_msg_id': 'B37FC4E0-FFB1-4645-A883-500BD5F663C0', 'text': 'I wanna maybe try some tests on this device before unloading if possible because I am just confused how the gate what show breakdown but then just like not work lol ', 'team': 'T0916MXAW2F', 'thread_ts': '1785331103.419059', 'reply_count': 1, 'reply_users_count': 1, 'latest_reply': '1785331138.624949', 'reply_users': ['U0BHW7P6D7C'], 'is_locked': False, 'blocks': [{'type': 'rich_text', 'block_id': 'ztttT', 'elements': [{'type': 'rich_text_section', 'elements': [{'type': 'text', 'text': 'I wanna maybe try some tests on this device before unloading if possible because I am just confused how the gate what show breakdown but then just like not work lol '}]}]}]}, 'ts': '1785331138.624949', 'client_msg_id': '20D7CC04-72F3-

INFO:slack_bolt.App:Connecting to a new endpoint...
INFO:slack_bolt.App:The connection has been closed (session id: 78b0a335-e654-4194-b336-66b6d8b40b98)
INFO:slack_bolt.App:A new session has been established (session id: 678bddf9-c276-4f77-a64e-18a18db9586d)
INFO:slack_bolt.App:Connected to a new endpoint...
INFO:slack_bolt.App:Stopped receiving messages from a connection (session id: 78b0a335-e654-4194-b336-66b6d8b40b98)
INFO:slack_bolt.App:Starting to receive messages from a new connection (session id: 678bddf9-c276-4f77-a64e-18a18db9586d)


In [26]:
studentbot.set_status('Hello. I am student bot.')

In [24]:
studentbot.stop()

INFO:slack_bolt.App:The connection has been closed (session id: 5512dc2e-4aa1-4cc4-995f-4dc4248907f2)


StudentBot stopped.


# Saving Data

Here we set up our QCoDeS database directory.

In [9]:
database_dir = r"C:\Users\KleinLab\Desktop\PPMS DATA\ET_hBNC_5"
database_name = "ET_hBNC_5"

db_path = os.path.join(database_dir, database_name)
initialise_or_create_database_at(db_path)

---

# Testing Gates

In [10]:
from qcodes import validators

Set gate voltage and bias limits

In [31]:
back_gate.source.voltage.remove_validator()
Vg_min = -8.05
Vg_max = 9.05
back_gate.source.voltage.add_validator(validators.Numbers(Vg_min, Vg_max))

In [32]:
print(back_gate.source.voltage.validators)

(<Numbers -8.05<=v<=9.05>,)


In [ ]:
# back_gate.source.voltage(0)

In [33]:
bias.source.voltage.remove_validator()
Vb_min = -0.55
Vb_max = 0.55
bias.source.voltage.add_validator(validators.Numbers(Vb_min, Vb_max))

In [34]:
print(bias.source.voltage.validators)

(<Numbers -0.55<=v<=0.55>,)


In [34]:
# bias.source.voltage(0)

# Scans

---

## MFLI-only $\frac{dI}{dV}$ Sweep 

The MFLI-only $\frac{dI}{dV}$ sweep is a sweep of the $V_{\rm DC}$ offset of the MFLI +$V$ signal output, while monitoring the demodulated $I$ signal input. 

#### Measurement Setup

In [197]:
### Lockin Parameters ###
# Reference Signal
ref_frequency = 17.7777777
output_amplitude_rms = 5e-3

# Low-pass Filter
filter_order = 4
filter_TC = 700e-3
##########################



# This is the most important parameter, but one that should not change in this cell. Here we set the sweep parameter, "param" to be 
# the DC V offset on the MFLI. 
# param = mfli.sigouts[0].offset # this is VDC offset
param = V_b

# Change these:
### Sweep Parameters ###
poll_time = 1 * filter_TC # we further average over poll_time
param_delay = 3 * filter_TC
start = -0.02
stop = 0.014
step_size = 0.0005 # step size might be slightly affected by np.linspace to get a whole number of steps
###########################

N_steps = int(np.abs(stop - start) / step_size)
param_range = np.linspace(start, stop, N_steps+1)
print(N_steps)
time_estimate = N_steps*(poll_time + param_delay)/60
print(f'estimated time {time_estimate}')

print(param_range)

68
estimated time 3.173333333333333
[-0.02   -0.0195 -0.019  -0.0185 -0.018  -0.0175 -0.017  -0.0165 -0.016
 -0.0155 -0.015  -0.0145 -0.014  -0.0135 -0.013  -0.0125 -0.012  -0.0115
 -0.011  -0.0105 -0.01   -0.0095 -0.009  -0.0085 -0.008  -0.0075 -0.007
 -0.0065 -0.006  -0.0055 -0.005  -0.0045 -0.004  -0.0035 -0.003  -0.0025
 -0.002  -0.0015 -0.001  -0.0005  0.      0.0005  0.001   0.0015  0.002
  0.0025  0.003   0.0035  0.004   0.0045  0.005   0.0055  0.006   0.0065
  0.007   0.0075  0.008   0.0085  0.009   0.0095  0.01    0.0105  0.011
  0.0115  0.012   0.0125  0.013   0.0135  0.014 ]


Then we send these parameters to the lockin. Sometimes you will get a `CoreError: Device DEVXXXXX timed out during multiple node get.` when you run the next cell. Just run the cell again. 

In [198]:
#set frequency
mfli.oscs[0].freq(ref_frequency)

# cursed next line due to some oversight in the zhinst drivers where the shortcut for .set() doesn't work for this specfici ZI node
# if mfli_serial == 'dev32203' or 'DEV32203': 
#     session.devices['dev32203'].sigouts[0].amplitudes[0].set(param_name='value', value=output_amplitude_rms*np.sqrt(2)) #sqrt 2 factor handles converting Vrms to Vpk
# if mfli_serial == 'dev32269' or 'DEV32269':
#     session.devices['dev32269'].sigouts[0].amplitudes[1].set(param_name='value', value=output_amplitude_rms*np.sqrt(2)) #sqrt 2 factor handles converting Vrms to Vpk

# Freaky AAAAAAAAAAA set amplitude 
if mfli_serial == 'dev32203' or 'DEV32203': 
     session.devices['dev32203'].sigouts[0].amplitudes[0].set(param_name='value', value=output_amplitude_rms*np.sqrt(2))

#set filter order, time constant
mfli.demods[0].order(filter_order)
mfli.demods[0].timeconstant(filter_TC)

#autorange output 
mfli.sigouts[0].autorange(1)

In [155]:
session.devices['dev32203'].demods[0].adcselect(1) #If you are measuring current channel
session.devices['dev32203'].currins[0].autorange(1)

In [199]:
mfli.sigouts[0].enables[0].set(param_name='value' , value = 1 ) # enable source 1 
mfli.sigouts[0].enables[1].set(param_name='value' , value = 0 ) # disable source 2
mfli.sigouts[0].enables[2].set(param_name='value' , value = 0 ) # disable source 3
mfli.sigouts[0].enables[3].set(param_name='value' , value = 0 ) # disable source 4

In [200]:
session.devices['dev32203'].demods[0].adcselect(0) #If you are measuring voltage channel
session.devices['dev32203'].sigins[0].autorange(1)

In [265]:
session.devices['dev32203'].sigouts[0].on(0)#Toggles ouput of lockin on and off

(2026 06 18 NP): As of now, we make you turn on signal output from the MFLI itself. Hopefully, opening LabOne means you will check that things are looking good before turning on signal output. 

Next we create a "poller" that will poll the MFLI for its data. This a bit of a gnarly hack fix from me to get this to work with QCoDeS, but it works and I CBA to do anything else. poller.get(poll_time=X) will poll the MFLI for X seconds and return an array of the time-averaged: [timestamp,
            x,
            y,
            frequency,
            phase,
            dio,
            auxin0,
            auxin1]
 in that order!

In [201]:
poller = MFLIDemodPoller("poller", zi_session=session, device=mfli, poll_time=poll_time)

#### Measurement Data Saving

Now we set up experiment name and sample name. Each experiment saves with a snapshot of the station in its metadata, so don't worry about saving MFLI metadata parameters. BUT, it is certainly convenient to save metadata parameters in the `experiment_name` / `sample_name`. 

In [229]:
experiment_name = "temp_dep_vb_sweeps"
additional_text = "linewidth_test_restart3"
# sample_name = f"{additional_text}_B{ppms.field.get()}T_T{ppms.temperature.get()}K_f{ref_frequency}Hz_Amp{output_amplitude_rms}V_TC{filter_TC}s_poll_time{poll_time}s_delay{param_delay}s_DivAC{v_div_AC_side_resistance}DC{v_div_DC_side_resistance}_PreAmp{preamp_gain}_Vb{V_b_range}V-res{V_b_step_size}V_Vg{V_g_range}V-{V_g_step_size}V"
sample_name = f"{additional_text}_B0T_T30K_f{ref_frequency}Hz_Vrms{output_amplitude_rms}V_TC{filter_TC}s_poll_time{poll_time}s_delay{param_delay}s_DivAC{v_div_AC_side_resistance}DC{v_div_DC_side_resistance}_PreAmp{preamp_gain}_Vb{[start, stop]}V-res{step_size}V"
print(sample_name)

linewidth_test_restart3_B0T_T30K_f17.7777777Hz_Vrms0.005V_TC0.7s_poll_time0.7s_delay2.0999999999999996s_DivAC40000.0DC10000.0_PreAmp1000.0_Vb[-0.02, 0.014]V-res0.0005V


In [230]:
exp = load_or_create_experiment(
    experiment_name = experiment_name,
    sample_name = sample_name
)

from qcodes.parameters import Parameter
# initialize qcodes measurement context for dIdV_sweep
meas = Measurement(exp=exp, name="dIdV")

meas.register_parameter(param)
meas.register_parameter(poller, setpoints=(param,))

# send parameters to monitor as well
monitor = qc.Monitor(param,mfli.demods[0].sample)
monitor.update_all()

meas.write_period = 0.1 # read from cache every X seconds

Below is the code to actually run the sweep. Before running the sweep we also create a .csv format truncated duplicate of the data for ease of use. However, all the data (including station metadata) is stored in the database.

In [204]:
log_text = f"""{additional_text}
Run id = ?

Lock-in Parameters 
Frequency = {ref_frequency:.3f} Hz
Pre-Divider Vrms = {output_amplitude_rms * 1000:.0f} mV
Vrms = {output_amplitude_rms * (v_div_DC_side_resistance / (v_div_AC_side_resistance + v_div_DC_side_resistance)) * 1000:.0f} mV
Vpk = {output_amplitude_rms * np.sqrt(2) * (v_div_DC_side_resistance / (v_div_AC_side_resistance + v_div_DC_side_resistance)) * 1000:.0f} mV
TC = {filter_TC * 1000:.0f} ms
Poll Time = {poll_time / filter_TC:g}*TC
Delay Time = {param_delay / filter_TC:g}*TC

Station Parameters
Voltage divider AC side resistance = {v_div_AC_side_resistance /1000:.0f} kOhm
Voltage divider DC side resistance = {v_div_DC_side_resistance /1000:.0f} kOhm
Femto Preamp Gain = {preamp_gain} V/A

PPMS Parameters
T = 2 K 
B = 0 T 

Sweep Parameters
Vb = [{start, stop}] V
Vb resolution = {step_size * 1000:.0f} mV

V_g = {V_g.get()}V
"""

print(log_text)

linewidth_test_restart3
Run id = ?

Lock-in Parameters 
Frequency = 17.778 Hz
Pre-Divider Vrms = 5 mV
Vrms = 1 mV
Vpk = 1 mV
TC = 700 ms
Poll Time = 1*TC
Delay Time = 3*TC

Station Parameters
Voltage divider AC side resistance = 40 kOhm
Voltage divider DC side resistance = 10 kOhm
Femto Preamp Gain = 1000.0 V/A

PPMS Parameters
T = 2 K 
B = 0 T 

Sweep Parameters
Vb = [(-0.02, 0.014)] V
Vb resolution = 0 mV

V_g = -2.7V



### <span style="color:red">**RUN SWEEP ↓**
</span>

Did you set current sense to 100 mA range? 

In [215]:
session.devices['dev32203'].sigouts[0].on(1) #Toggles ouput of lockin on and of
session.devices['dev32203'].sigins[0].autorange(1)
# session.devices['dev32203'].currins[0].autorange(1)

In [231]:
# csv_dir = os.path.join(f'{db_path} Data', experiment_name)
# os.makedirs(csv_dir, exist_ok=True)
# studentbot.set_status('Running 1D Sweep.')
with meas.run() as datasaver:
    # ↓ main scan loop ↓
    time.sleep(param_delay)
    for itr, param_value in enumerate(tqdm(param_range)):

        ramp(param=param, setpoint=param_value, slew_rate=0.05, param_increment=.01) 
        time.sleep(param_delay)
        get_poll = poller.get()

        datasaver.add_result(
            (param, param.get()), 
            (poller, get_poll)
        )

        #monitor.update_all()
        
    ##########################
    # studentbot.set_status('Finished 1D Sweep.')        
    dataset1D = datasaver.dataset
    df = dataset1D.to_pandas_dataframe()
    
    # csv_filename = f'{sample_name}_{datasaver.run_id}.csv'
    # csv_path = os.path.join(csv_dir, csv_filename)
    # df.to_csv(csv_path)

Starting experimental run with id: 32. 


100%|██████████| 69/69 [03:18<00:00,  2.87s/it]


In [191]:
ramp(V_g,-2.7,slew_rate=0.1,param_increment=0.01,verbose=True)

done ramping


#### RC Time Constant Testing

In [284]:
### Lockin Parameters ###
# Reference Signal
ref_frequency = 17.7777777
output_amplitude_rms = 7.7e-3

# Low-pass Filter
filter_order = 4
filter_TC = 700e-3
##########################


poll_time = 100e-3 #Set the data sampling time

initial_bias = -0.040
final_bias = -0.038

N_poll = 300 # Number of times you poll after setting the bias 

param = V_b

##Setting Lockin to chosen parameters

#Set frequency 
mfli.oscs[0].freq(ref_frequency)

#set amplitude 
if mfli_serial == 'dev32203' or 'DEV32203': 
     session.devices['dev32203'].sigouts[0].amplitudes[0].set(param_name='value', value=output_amplitude_rms*np.sqrt(2))

#set filter order, time constant
mfli.demods[0].order(filter_order)
mfli.demods[0].timeconstant(filter_TC)

#autorange output 
mfli.sigouts[0].autorange(1)

mfli.sigouts[0].enables[0].set(param_name='value' , value = 1 ) # enable source 1 
mfli.sigouts[0].enables[1].set(param_name='value' , value = 0 ) # disable source 2
mfli.sigouts[0].enables[2].set(param_name='value' , value = 0 ) # disable source 3
mfli.sigouts[0].enables[3].set(param_name='value' , value = 0 ) # disable source 4

session.devices['dev32203'].demods[0].adcselect(0) #If you are measuring voltage channel
session.devices['dev32203'].sigins[0].autorange(1)

#set up poller 
poller = MFLIDemodPoller("poller", zi_session=session, device=mfli, poll_time=poll_time)

In [290]:
experiment_name = "sweep_optimization"
sample_name = "RC_time_const_2mVstep_repeat_B0T_T1p8K_Vgn1p5V_f17p777Hz_Amp2mV_TC800ms_Poll100ms_Npoll300_Div4-5_PreAmp_10^4"

In [291]:
exp = load_or_create_experiment(
    experiment_name = experiment_name,
    sample_name = sample_name
)

from qcodes.parameters import Parameter
# initialize qcodes measurement context for dIdV_sweep
meas = Measurement(exp=exp, name="dIdV")

meas.register_parameter(param)
meas.register_parameter(poller, setpoints=(param,))

# send parameters to monitor as well
monitor = qc.Monitor(param,mfli.demods[0].sample)
monitor.update_all()

meas.write_period = 0.1 # read from cache every X seconds

In [292]:
session.devices['dev32203'].sigouts[0].on(1) #Toggles ouput of lockin on and of
session.devices['dev32203'].sigins[0].autorange(1) #Autoranges Lockin Input 

ramp(V_b,initial_bias,slew_rate=0.1,param_increment=0.01,verbose=True)
time.sleep(10)
print('Setting bias')
ramp(V_b,final_bias,slew_rate=0.1,param_increment=0.01,verbose=True)
with meas.run() as datasaver:
    for i in range(N_poll):
        get_poll = poller.get()
        monitor.update_all()
        datasaver.add_result(
            (param, param.get()), 
            (poller, get_poll)
        )

Already at setpoint
Setting bias
Already at setpoint
Starting experimental run with id: 28. 


#### EXAMPLE: Loading station metadata

In [24]:
from qcodes.dataset import load_by_id
dataset = load_by_id(run_id=1)
print(dataset.snapshot)

{'station': {'instruments': {'zi_baseinstrument_dev32269': {'functions': {}, 'submodules': {'stats': {'functions': {}, 'submodules': {'physical': {'functions': {}, 'submodules': {'temperatures': {'channels': {'zi_baseinstrument_dev32269_stats_physical_temperatures0': {'functions': {}, 'submodules': {}, 'parameters': {'value': {'__class__': 'zhinst.qcodes.qcodes_adaptions.ZIParameter', 'full_name': 'zi_baseinstrument_dev32269_stats_physical_temperatures0_value', 'value': None, 'raw_value': None, 'ts': None, 'unit': '°C', 'name': 'value', 'inter_delay': 0, 'label': 'value', 'post_delay': 0, 'validators': [], 'instrument': 'zhinst.qcodes.qcodes_adaptions.ZINode', 'instrument_name': 'zi_baseinstrument_dev32269_stats_physical_temperatures0'}}, '__class__': 'zhinst.qcodes.qcodes_adaptions.ZINode', 'name': 'zi_baseinstrument_dev32269_stats_physical_temperatures0', 'label': 'temperatures0'}, 'zi_baseinstrument_dev32269_stats_physical_temperatures1': {'functions': {}, 'submodules': {}, 'paramet

## ($V_b$, $V_g$) Sweep



**Sweep Parameters**

(slow) $V_g$ -> Keithley 1

(fast) $V_b$ -> Keithley 2

**Dependent Parameters**

$\frac{dI}{dV}$ -> MFLI 1



#### Measurement Setup

First, we set up the Keithleys.

Before we do anything, it's probably smart to safely ramp to 0V on both Keithleys. I wrote `ramp` up in the first cell if you want to see its arguments. 

In [233]:
ramp(V_b, 0,slew_rate=0.2,param_increment=0.01,verbose=True)
ramp(V_g, 0,slew_rate=0.2,param_increment=0.01,verbose=True)


Already at setpoint
Already at setpoint


Then, let's set lockin parameters.

In [162]:
### Lockin Parameters ###
# (edit these)

# reference signal
ref_frequency = 17.7777777
output_amplitude_rms = 10e-3

# low-pass filter
filter_order = 4
filter_TC = 300e-3

# polling 
poll_time = 1 * filter_TC # we further average over poll_time
param_delay = 3 * filter_TC # delay between setting voltage and start of lockin poll
##########################


mfli.oscs[0].freq(ref_frequency)
# cursed next line due to some oversight in the zhinst drivers where the shortcut for .set() doesn't work for this specfici ZI node
if mfli_serial == 'dev32203' or 'DEV32203': 
     session.devices['dev32203'].sigouts[0].amplitudes[0].set(param_name='value', value=output_amplitude_rms*np.sqrt(2)) #sqrt 2 factor handles converting Vrms to Vpk
#if mfli_serial == 'dev32269' or 'DEV32269':
#    session.devices['dev32269'].sigouts[0].amplitudes[1].set(param_name='value', value=output_amplitude_rms*np.sqrt(2)) #sqrt 2 factor handles converting Vrms to Vpk
mfli.demods[0].order(filter_order)
mfli.demods[0].timeconstant(filter_TC)
mfli.sigouts[0].autorange(1)

mfli.sigouts[0].enables[0].set(param_name='value' , value = 1 ) # enable source 1 
mfli.sigouts[0].enables[1].set(param_name='value' , value = 0 ) # disable source 2
mfli.sigouts[0].enables[2].set(param_name='value' , value = 0 ) # disable source 3
mfli.sigouts[0].enables[3].set(param_name='value' , value = 0 ) # disable source 4

session.devices['dev32203'].demods[0].adcselect(0) #If you are measuring voltage channel
session.devices['dev32203'].sigins[0].autorange(1)

poller = MFLIDemodPoller("poller", zi_session=session, device=mfli, poll_time=poll_time)

Now let's set the parameters of our sweep. 

In [166]:
### Sweep Parameters ###
# (edit these)

V_b_range = [-0.4, 0.4]
V_g_range = [-7, 3]

V_g_step_size = 0.1
V_b_step_size = 0.01

### Ramp Parameters ### 
# when stepping between values in the sweep, these parameters control the ramp() method defined in the first cell
# (edit these)
slew_rate = 0.05
param_increment = 0.005
min_inter_delay = 0.005

slow_param = V_g
fast_param = V_b

slow_param_start_stop = V_g_range
fast_param_start_stop = V_b_range

N_steps_slow = int(np.abs(slow_param_start_stop[0] - slow_param_start_stop[-1]) / V_g_step_size)
N_steps_fast = int(np.abs(fast_param_start_stop[0] - fast_param_start_stop[-1]) / V_b_step_size)

print(f'Number of Vb sweeps: {N_steps_slow}')
print(f'Number of points in each sweep: {N_steps_fast}')

time_estimate = N_steps_slow*N_steps_fast*(poll_time+param_delay)/60/60*1.2
print(f'Estimated time: {time_estimate} hours')

slow_param_range = np.linspace(slow_param_start_stop[0], slow_param_start_stop[-1], N_steps_slow+1)
fast_param_range = np.linspace(fast_param_start_stop[0], fast_param_start_stop[-1], N_steps_fast+1)
# print((slow_param_range))
# print(N_steps_slow)

Number of Vb sweeps: 100
Number of points in each sweep: 80
Estimated time: 3.1999999999999997 hours


#### Measurement Data Saving

The sweep parameters are now set up, we just need to set our experiment and sample name. Each experiment is saved alongside the full state of the QCoDeS station, so you don't have to worry about saving those manually, however it might be easier than parsing the huge .json file that you'll get.

In [167]:
experiment_name = "mining_diamonds_after_reload"
additional_text = "map4_Bfield_expanded_coarse"
# sample_name = f"{additional_text}_B{ppms.field.get()}T_T{ppms.temperature.get()}K_f{ref_frequency}Hz_Amp{output_amplitude_rms}V_TC{filter_TC}s_poll_time{poll_time}s_delay{param_delay}s_DivAC{v_div_AC_side_resistance}DC{v_div_DC_side_resistance}_PreAmp{preamp_gain}_Vb{V_b_range}V-res{V_b_step_size}V_Vg{V_g_range}V-{V_g_step_size}V"
sample_name = f"{additional_text}_B9T_T2K_f{ref_frequency}Hz_Vrms{output_amplitude_rms}V_TC{filter_TC}s_poll_time{poll_time}s_delay{param_delay}s_DivAC{v_div_AC_side_resistance}DC{v_div_DC_side_resistance}_PreAmp{preamp_gain}_Vb{V_b_range}V-res{V_b_step_size}V_Vg{V_g_range}V-{V_g_step_size}V"
print(sample_name)

map4_Bfield_expanded_coarse_B9T_T2K_f17.7777777Hz_Vrms0.01V_TC0.3s_poll_time0.3s_delay0.8999999999999999s_DivAC40000.0DC10000.0_PreAmp1000.0_Vb[-0.4, 0.4]V-res0.01V_Vg[-7, 3]V-0.1V


In [168]:
exp = load_or_create_experiment(
    experiment_name = experiment_name,
    sample_name = sample_name
)
from qcodes.parameters import Parameter

meas = Measurement(exp=exp, name="(V_b,V_g)_sweep")
meas.register_parameter(slow_param)
meas.register_parameter(fast_param)
# meas.register_parameter(ppms.temperature, setpoints=(slow_param, fast_param))
meas.register_parameter(poller, setpoints=(slow_param, fast_param))

# meas.register_parameter(keithley_2.sense.current, setpoints=(slow_param, fast_param))

# # send parameters to monitor as well
# monitor = qc.Monitor(V_g, V_b, mfli.demods[0].sample)
# monitor.update_all()

meas.write_period = 0.1 # read from cache every X seconds

In [169]:
log_text = f"""{additional_text}
Run id = ?

Lock-in Parameters 
Frequency = {ref_frequency:.3f} Hz
Pre-Divider Vrms = {output_amplitude_rms * 1000:.0f} mV
Vrms = {output_amplitude_rms * (v_div_DC_side_resistance / (v_div_AC_side_resistance + v_div_DC_side_resistance)) * 1000:.0f} mV
Vpk = {output_amplitude_rms * np.sqrt(2) * (v_div_DC_side_resistance / (v_div_AC_side_resistance + v_div_DC_side_resistance)) * 1000:.0f} mV
TC = {filter_TC * 1000:.0f} ms
Poll Time = {poll_time / filter_TC:g}*TC
Delay Time = {param_delay / filter_TC:g}*TC

Station Parameters
Voltage divider AC side resistance = {v_div_AC_side_resistance /1000:.0f} kOhm
Voltage divider DC side resistance = {v_div_DC_side_resistance /1000:.0f} kOhm
Femto Preamp Gain = {preamp_gain} V/A

PPMS Parameters
T = 2 K 
B = 9 T 

Sweep Parameters
Vb = {V_b_range} V
Vb resolution = {V_b_step_size * 1000:.0f} mV
Vg = {V_g_range} V
Vg resolution = {V_g_step_size * 1000:.0f} mV"""

print(log_text)

map4_Bfield_expanded_coarse
Run id = ?

Lock-in Parameters 
Frequency = 17.778 Hz
Pre-Divider Vrms = 10 mV
Vrms = 2 mV
Vpk = 3 mV
TC = 300 ms
Poll Time = 1*TC
Delay Time = 3*TC

Station Parameters
Voltage divider AC side resistance = 40 kOhm
Voltage divider DC side resistance = 10 kOhm
Femto Preamp Gain = 1000.0 V/A

PPMS Parameters
T = 2 K 
B = 9 T 

Sweep Parameters
Vb = [-0.4, 0.4] V
Vb resolution = 10 mV
Vg = [-7, 3] V
Vg resolution = 100 mV


### <span style="color:red">**RUN SWEEP ↓**
</span>

In [170]:
session.devices['dev32203'].sigouts[0].on(1) #Toggles ouput of lockin on and of
session.devices['dev32203'].sigins[0].autorange(1)
# session.devices['dev32203'].currins[0].autorange(1)

In [171]:
csv_dir = os.path.join(f'{db_path} csv data', experiment_name)
os.makedirs(csv_dir, exist_ok=True)

with meas.run() as datasaver:
    
    for slow_itr, slow_param_value in enumerate(tqdm(slow_param_range)):
        # print(f"[PPMS] Field: {ppms.field.get()} Oe | Temperature {ppms.temperature.get()} K")
        # studentbot.set_status(f"Sweep {round((slow_itr+1) / N_steps_slow, 2)}% complete. Vg = {slow_param_value} V")
        print(f"[Gate] Ramping from {slow_param.get()} to {slow_param_value}V.")
        ramp(param=slow_param, setpoint=slow_param_value, slew_rate=slew_rate, param_increment=param_increment, min_inter_delay=min_inter_delay)
        print(f"[Gate] Ramped to {slow_param_value}V. Ramping to fast sweep value to {fast_param_range[0]}V.")
        for fast_itr, fast_param_value in enumerate(fast_param_range):
            ramp(param=fast_param, setpoint=fast_param_value, slew_rate=slew_rate, param_increment=param_increment, min_inter_delay=min_inter_delay)
            time.sleep(param_delay)
            get_poll = poller.get()
            #gate_current = keithley_2.sense.current()
            datasaver.add_result(
                (slow_param, slow_param_value),
                (fast_param, fast_param_value),
                # (keithley_2.sense.current, gate_current),
                (poller, get_poll)
                # (ppms.temperature, ppms.temperature.get())
            )
            time.sleep(min_inter_delay)
            #monitor.update_all()
            #datasaver.flush_data_to_database()
    # studentbot.set_status("Sweep complete :)")        
    dataset1D = datasaver.dataset
    df = dataset1D.to_pandas_dataframe()
    csv_filename = f'{sample_name}_{datasaver.run_id}.csv'
    csv_path = os.path.join(csv_dir, csv_filename)
    df.to_csv(csv_path)

Starting experimental run with id: 20. 


  0%|          | 0/101 [00:00<?, ?it/s]

[Gate] Ramping from -5.0 to -7.0V.
[Gate] Ramped to -7.0V. Ramping to fast sweep value to -0.4V.


  1%|          | 1/101 [02:45<4:35:19, 165.19s/it]

[Gate] Ramping from -7.0 to -6.9V.
[Gate] Ramped to -6.9V. Ramping to fast sweep value to -0.4V.


...

 98%|█████████▊| 99/101 [3:28:51<04:12, 126.17s/it]

[Gate] Ramping from 2.8 to 2.9000000000000004V.
[Gate] Ramped to 2.9000000000000004V. Ramping to fast sweep value to -0.4V.


 99%|█████████▉| 100/101 [3:30:57<02:06, 126.16s/it]

[Gate] Ramping from 2.9 to 3.0V.
[Gate] Ramped to 3.0V. Ramping to fast sweep value to -0.4V.


100%|██████████| 101/101 [3:33:03<00:00, 126.57s/it]


In [173]:
ramp(V_b, 0,slew_rate=0.2,param_increment=0.01,verbose=True)
ramp(V_g, 0,slew_rate=0.2,param_increment=0.01,verbose=True)

Already at setpoint
Already at setpoint


#### Dummy Sweep 

In [242]:
experiment_name = "dummy"
sample_name = 'dummy'

exp = load_or_create_experiment(
    experiment_name = experiment_name,
    sample_name = sample_name
)

meas = Measurement(exp=exp, name="dumb")
meas.register_parameter(V_b)

In [243]:
with meas.run() as datasaver:
    print('dummy for loading data')

Starting experimental run with id: 37. 
dummy for loading data


## $(V_g, B)$ Sweep


**Sweep Parameters**

(fast) $V_g$ -> Keithley 1

(slow) $B$ -> PPMS

**Dependent Parameters**

$\frac{dI}{dV}$ -> MFLI 1




In [35]:
### Lockin Parameters ###
# (edit these)

# reference signal
ref_frequency = 17.7777777
output_amplitude_rms = 7.7e-3

# low-pass filter
filter_order = 4
filter_TC = 500e-3

# polling 
poll_time = 1 * filter_TC # we further average over poll_time

param_delay = 3 * filter_TC
##########################


mfli.oscs[0].freq(ref_frequency)
# cursed next line due to some oversight in the zhinst drivers where the shortcut for .set() doesn't work for this specfici ZI node
if mfli_serial == 'dev32203' or 'DEV32203': 
     session.devices['dev32203'].sigouts[0].amplitudes[0].set(param_name='value', value=output_amplitude_rms*np.sqrt(2)) #sqrt 2 factor handles converting Vrms to Vpk
#if mfli_serial == 'dev32269' or 'DEV32269':
#    session.devices['dev32269'].sigouts[0].amplitudes[1].set(param_name='value', value=output_amplitude_rms*np.sqrt(2)) #sqrt 2 factor handles converting Vrms to Vpk
mfli.demods[0].order(filter_order)
mfli.demods[0].timeconstant(filter_TC)
mfli.sigouts[0].autorange(1)

mfli.sigouts[0].enables[0].set(param_name='value' , value = 1 ) # enable source 1 
mfli.sigouts[0].enables[1].set(param_name='value' , value = 0 ) # disable source 2
mfli.sigouts[0].enables[2].set(param_name='value' , value = 0 ) # disable source 3
mfli.sigouts[0].enables[3].set(param_name='value' , value = 0 ) # disable source 4

session.devices['dev32203'].demods[0].adcselect(0) #If you are measuring voltage channel
session.devices['dev32203'].sigins[0].autorange(1)

poller = MFLIDemodPoller("poller", zi_session=session, device=mfli, poll_time=poll_time)

In [36]:
B_rate = 30
B_approach_mode = 0 # 0 is linear
B_end_mode = 0 # persistent

In [64]:
### Sweep Parameters ###
# (edit these)
B_range = [3409.090909090908, 0]
V_g_range = [-3, 2.5]

B_step_size = 900
V_g_step_size = 0.05

### B Ramp Parameters ###
total_time_allowed = 600 # seconds, total time we allow to wait for next magnet setpoint to reach persistent
B_wait_time = 30 # seconds, time between checks that the magnet controller field is persistent

N_tries_for_field = int(total_time_allowed/wait_time)

### Voltage Ramp Parameters ### 
# when stepping between values in the sweep, these parameters control the ramp() method defined in the first cell
# (edit these)
slew_rate = 0.1
param_increment = 0.05
min_inter_delay = 0.005

B_wait_time_estimate = 4 * 30

slow_param = B
fast_param = V_g

slow_param_start_stop = B_range
fast_param_start_stop = V_g_range

N_steps_slow = int(np.abs(B_range[0] - B_range[-1]) / B_step_size)
N_steps_fast = int(np.abs(fast_param_start_stop[0] - fast_param_start_stop[-1]) / V_g_step_size)

print(f'Number of points in each sweep: {N_steps_fast}')
print(f'Number of B points: {N_steps_slow}')

time_estimate = B_wait_time_estimate*N_steps_slow/60/60*1.2 + N_steps_slow*N_steps_fast*(poll_time+param_delay)/60/60*1.2
print(f'Estimated time: {time_estimate} hours')

slow_param_range = np.linspace(slow_param_start_stop[0], slow_param_start_stop[-1], N_steps_slow)
fast_param_range = np.linspace(fast_param_start_stop[0], fast_param_start_stop[-1], N_steps_fast)

Number of points in each sweep: 110
Number of B points: 3
Estimated time: 0.33999999999999997 hours


In [37]:
ramp(V_b,0,slew_rate=0.1,param_increment=0.01,verbose=True)

done ramping


In [65]:
experiment_name = "Vg_B_maps"
sample_name = "sweep1_continue_after_error2_B9to0T_T1p8K_f17p777Hz_Amp2mV_TC500ms_poll1_delay3_Div4-5_PreAmp10^4_Vbn0.6_Vgn3to2p5V-res50mV"

In [66]:
from qcodes.parameters import Parameter

exp = load_or_create_experiment(
    experiment_name = experiment_name,
    sample_name = sample_name
)

meas = Measurement(exp=exp, name="(V_g,B)_sweep")
meas.register_parameter(slow_param)
meas.register_parameter(fast_param)
meas.register_parameter(poller, setpoints=(slow_param, fast_param))
# meas.register_parameter(keithley_2.sense.current, setpoints=(slow_param, fast_param))

meas.write_period = 0.1 # read from cache every X seconds

In [67]:
with meas.run() as datasaver:
    # B-specific slow parameter loop
    for slow_itr, slow_param_value in enumerate(tqdm(slow_param_range)):
        
        ppms.client.set_field(slow_param_value, B_rate, B_approach_mode, B_end_mode)
        for i in range(N_tries_for_field):
            if i == N_tries_for_field - 1:
                raise Exception(f'Did not reach field setpoint in {total_time_allowed/60} min. Raising exception and stopping scan.')
            time.sleep(B_wait_time)
            if ppms.field_status.get() == "Persistent":
                print(f'[PPMS Status] (T={ppms.temperature.get()}K) Magnet: Persistent | Reached {slow_param_value} Oe in {i+1} {B_wait_time}s intervals.')
                break
            else:
                print(f'[PPMS Status] (T={ppms.temperature.get()}K) Magnet: {ppms.field_status.get()}')
                continue
                
        # fast parameter same as usual        
        for fast_itr, fast_param_value in enumerate(fast_param_range):
            ramp(param=fast_param, setpoint=fast_param_value, slew_rate=slew_rate, param_increment=param_increment, min_inter_delay=min_inter_delay)
            time.sleep(param_delay)
            get_poll = poller.get()
            #gate_current = keithley_2.sense.current()
            datasaver.add_result(
                (slow_param, slow_param_value),
                (fast_param, fast_param_value),
                # (keithley_2.sense.current, gate_current),
                (poller, get_poll)
            )
            time.sleep(min_inter_delay)

        print('[Gate] Ramping gate voltage to initial setpoint.')
        ramp(V_g, fast_param_range[0],slew_rate=0.2,param_increment=0.01,verbose=True)

Starting experimental run with id: 56. 


  0%|          | 0/3 [00:00<?, ?it/s]

[PPMS Status] (T=1.799K) Magnet: Switch cooling
[PPMS Status] (T=1.8006K) Magnet: Persistent | Reached 3409.090909090908 Oe in 2 30s intervals.
[Gate] Ramping gate voltage to initial setpoint.


 33%|███▎      | 1/3 [05:46<11:32, 346.36s/it]

done ramping
[PPMS Status] (T=1.8015K) Magnet: Status code not in dictionary
[PPMS Status] (T=1.7995K) Magnet: Status code not in dictionary
[PPMS Status] (T=1.8012K) Magnet: Iterating
[PPMS Status] (T=1.7997K) Magnet: Persistent | Reached 1704.545454545454 Oe in 4 30s intervals.
[Gate] Ramping gate voltage to initial setpoint.


 67%|██████▋   | 2/3 [12:01<06:03, 363.54s/it]

done ramping
[PPMS Status] (T=1.8004K) Magnet: Status code not in dictionary
[PPMS Status] (T=1.7998K) Magnet: Status code not in dictionary
[PPMS Status] (T=1.8004K) Magnet: Switch cooling
[PPMS Status] (T=1.7997K) Magnet: Persistent | Reached 0.0 Oe in 4 30s intervals.
[Gate] Ramping gate voltage to initial setpoint.


100%|██████████| 3/3 [18:17<00:00, 365.85s/it]

done ramping


In [81]:
ppms.temperature()

167.57

In [89]:
for i in range(60):
    print(f"[{round(i * 30 / 60, 2)} min] PPMS temperature: {ppms.temperature()} K")
    time.sleep(30)

[0.0 min] PPMS temperature: 9.908 K
[0.5 min] PPMS temperature: 8.3786 K
[1.0 min] PPMS temperature: 6.5194 K
[1.5 min] PPMS temperature: 4.6735 K
[2.0 min] PPMS temperature: 3.6444 K
[2.5 min] PPMS temperature: 3.1352 K
[3.0 min] PPMS temperature: 2.8356 K
[3.5 min] PPMS temperature: 2.6455 K
[4.0 min] PPMS temperature: 2.5101 K
[4.5 min] PPMS temperature: 2.4093 K
[5.0 min] PPMS temperature: 2.3333 K
[5.5 min] PPMS temperature: 2.2724 K
[6.0 min] PPMS temperature: 2.2231 K
[6.5 min] PPMS temperature: 2.1832 K
[7.0 min] PPMS temperature: 2.1484 K
[7.5 min] PPMS temperature: 2.1188 K
[8.0 min] PPMS temperature: 2.0937 K
[8.5 min] PPMS temperature: 2.0718 K
[9.0 min] PPMS temperature: 2.053 K
[9.5 min] PPMS temperature: 2.0349 K
[10.0 min] PPMS temperature: 2.0185 K
[10.5 min] PPMS temperature: 2.0043 K
[11.0 min] PPMS temperature: 1.9912 K
[11.5 min] PPMS temperature: 1.9793 K
[12.0 min] PPMS temperature: 1.9681 K
[12.5 min] PPMS temperature: 1.9582 K
[13.0 min] PPMS temperature: 1.948

In [128]:
ppms.temperature.get()

124.8517

In [129]:
ppms.temperature_status.get()

'Standby'